In [ ]:
import os
import sys
paths = ['/home/aiuser/work/workspace/BigAlpha/system/alphathonapiserver']
for path in paths:
    if path not in sys.path:
        sys.path.append(path)
from judge.judgebase import JudgeBase


JUDGE_RUNNER_CODE = '''
__USER_CODE__

def judge_runner_main():
    import pandas as pd

    data = main("cpt_jyc_2025_stock_csi1000_bar1m_test", "2025-01-01", "2025-07-31 23:59:59")

    from bigmodule import M
    result = M.bigalpha_factorminer._latest(
        factor_data=factor_data,
        factor_pool=factor_pool,
        process_pools=False,
        show=True,
    )

    with open("output.data", "w") as writer:
        writer.write(result._result.id)
'''


class Judge(JudgeBase):
    competition_id = "76ad3f56-ec2b-431a-890e-139a7f4bbcba"
    mode = "public"
    JUDGE_RUNNER_CODE = JUDGE_RUNNER_CODE

    def compute_score(self, df):
        df["score"] = (
            df["rank_ic"].rank(pct=True) * 0.4
            + df["rank_ir"].rank(pct=True) * 0.3
            + df["sharp_ratio"].rank(pct=True) * 0.2
            + df["turnover"].rank(pct=True, ascending=False) * 0.1
        )
        return df


judge = Judge()
# judge.run()

In [4]:
submissions = judge.alphathon_api.query_submissions(
    competition_id=judge.competition_id,
    constraints=judge.query_constraints(),
)

dispatched: set[str] = set()
pending = [s for s in submissions if s.get("id") not in dispatched]
submission = pending[0]
submission

2026-06-17 11:35:11 [info     ] HTTP Request: GET http://alphathonapiserver.bigquant.svc.cluster.local:8000/bigapis/alphathon/v1/submissions?competition_id=76ad3f56-ec2b-431a-890e-139a7f4bbcba&page=1&size=5000&order_by=-created_at&constraints=%7B%7D "HTTP/1.1 200 OK"


{'id': '0c9c3d39-24cc-467f-860a-ec967f901b4f',
 'created_at': '2026-06-17T11:29:33.301167+08:00',
 'updated_at': '2026-06-17T11:29:41.100704+08:00',
 'competition_id': '76ad3f56-ec2b-431a-890e-139a7f4bbcba',
 'user_id': '5dd35480-0f38-11ed-93bb-da75731aa77c',
 'data': {'files': {'15fe17b53ce74cdab70444e304c02e2e': {'name': 'testmd.md',
    'size': 4},
   '2ce0c7f96fd04c41aa6550328b0e628e': {'name': 'factor_sql.ipynb',
    'size': 3771}},
  'description': ''},
 'public_score': '-2.00000',
 'public_score_data': {'err_msg': 'run error: check your code / get code templates in [code] tab'},
 'private_score': None,
 'private_score_data': {},
 'selected_for_private': False}

In [7]:
for submission in pending:
    judge.save_submission_files(submission)
    runner = judge.run_user_code(submission)

2026-06-17 11:44:49 [info     ] HTTP Request: GET http://alphathonapiserver.bigquant.svc.cluster.local:8000/bigapis/alphathon/v1/submissions/files/0c9c3d39-24cc-467f-860a-ec967f901b4f/15fe17b53ce74cdab70444e304c02e2e "HTTP/1.1 200 OK"
2026-06-17 11:44:49 [info     ] HTTP Request: GET http://alphathonapiserver.bigquant.svc.cluster.local:8000/bigapis/alphathon/v1/submissions/files/0c9c3d39-24cc-467f-860a-ec967f901b4f/2ce0c7f96fd04c41aa6550328b0e628e "HTTP/1.1 200 OK"
2026-06-17 11:44:49 [info     ] submission.files_saved         count=2 dir=/home/aiuser/work/workspace/BigAlpha/system/files/76ad3f56-ec2b-431a-890e-139a7f4bbcba/submissions/0c9c3d39-24cc-467f-860a-ec967f901b4f submission_id=0c9c3d39-24cc-467f-860a-ec967f901b4f


Exception: submission 0c9c3d39-24cc-467f-860a-ec967f901b4f has 2 files, while only 1 is expected